# Borůvka minimum spanning forest (PySpark)

This notebook runs the same pipeline as `run_mst.py`: load graph text files under `data/`, execute Borůvka’s algorithm on Spark RDDs, and inspect the MSF/MST. Each section also writes results under the project `output/` folder (same format as the CLI). After the Spark run, a **Kruskal validation** step compares `total_weight`, `num_components`, and `|MST|` to a driver-side reference (see §“Validation”).

**Prerequisites:** a JDK, `pip install -r requirements.txt`, and a Jupyter environment (`pip install jupyter ipykernel`).

Set the kernel’s **working directory** to this project folder, or keep the default cells below—they locate the repo by searching for `src/boruvka_spark.py`.

In [ ]:
# pip install -r requirements.txt

In [10]:
import os
import sys
from pathlib import Path

# Resolve project root whether the kernel cwd is the repo or a subfolder (e.g. notebooks/).
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJ_ROOT = next(
    (p for p in _candidates if (p / "src" / "boruvka_spark.py").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJ_ROOT))
os.chdir(PROJ_ROOT)
print("PROJ_ROOT =", PROJ_ROOT.resolve())

PROJ_ROOT = /Users/kentpun/Documents/_04.studies/MSc BDT/MSBD 5003/_05. Project/Implementation


In [11]:
from pyspark.sql import SparkSession

from src.boruvka_spark import BoruvkaResult, boruvka_mst, run_boruvka_from_file
from src.mst_output import write_mst_file

spark = (
    SparkSession.builder.appName("BoruvkaMST-Notebook")
    .master(os.environ.get("SPARK_MASTER", "local[*]"))
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark ready")

Spark ready


## Validation (Kruskal reference)

The next cell defines `validate_vs_kruskal`, which loads the same graph as Spark (or takes an explicit edge list), runs **Kruskal** on the driver, and checks `total_weight`, `num_components`, and `|MST|`. MST edge *sets* can differ when weights tie; those three checks still certify a correct MSF weight and structure.

In [12]:
from src.graph_loader import load_graph_from_text
from src.mst_reference import kruskal_msf


def validate_vs_kruskal(
    boruvka_res: BoruvkaResult,
    *,
    edges_path: Path | None = None,
    fmt: str = "edge_list",
    edges: list[tuple[int, int, int]] | None = None,
    name: str = "graph",
) -> None:
    if edges is not None:
        e = edges
    elif edges_path is not None:
        e = load_graph_from_text(
            edges_path.read_text(encoding="utf-8"), fmt=fmt
        )
    else:
        raise ValueError("Provide edges_path or edges=")

    ref_w, ref_mst, ref_comp = kruskal_msf(e)
    assert boruvka_res.total_weight == ref_w, (
        f"{name}: total_weight spark={boruvka_res.total_weight} kruskal={ref_w}"
    )
    assert boruvka_res.num_components == ref_comp, (
        f"{name}: components spark={boruvka_res.num_components} kruskal={ref_comp}"
    )
    assert len(boruvka_res.mst_edges) == len(ref_mst), (
        f"{name}: |MST| spark={len(boruvka_res.mst_edges)} kruskal={len(ref_mst)}"
    )
    print(
        f"✓ {name}: Kruskal agrees — weight={ref_w}, components={ref_comp}, |MST|={len(ref_mst)}"
    )

## 1. Edge list (`data/sample_edges.txt`)

Triangle graph: expect MST edges `(0,1,1)` and `(1,2,2)`, total weight `3`.

In [13]:
edge_path = PROJ_ROOT / "data" / "sample_edges.txt"
res = run_boruvka_from_file(sc, str(edge_path), fmt="edge_list")

out_path = write_mst_file(res, PROJ_ROOT / "output" / "mst_sample_edges.txt", source=str(edge_path))
print("Wrote", out_path)

print("MST edges (u, v, w):", res.mst_edges)
print("total_weight =", res.total_weight)
print("num_iterations =", res.num_iterations)
print("num_components =", res.num_components)

validate_vs_kruskal(res, edges_path=edge_path, fmt="edge_list", name="§1 sample_edges")

Wrote /Users/kentpun/Documents/_04.studies/MSc BDT/MSBD 5003/_05. Project/Implementation/output/mst_sample_edges.txt
MST edges (u, v, w): [(0, 1, 1), (1, 2, 2)]
total_weight = 3
num_iterations = 1
num_components = 1
✓ §1 sample_edges: Kruskal agrees — weight=3, components=1, |MST|=2


### MST as a table

In [14]:
import pandas as pd

pd.DataFrame(res.mst_edges, columns=["u", "v", "weight"])

,u,v,weight
0,0,1,1
1,1,2,2


## 2. Build an RDD (`boruvka_mst`)

Same as the CLI, this constructs `edges_rdd` from any source instead of a file.

In [15]:
inline_edges = [(0, 1, 4), (0, 2, 8), (1, 2, 2), (1, 3, 6), (2, 3, 3)]
edges_rdd = sc.parallelize(inline_edges)
res2 = boruvka_mst(sc, edges_rdd)
write_mst_file(res2, PROJ_ROOT / "output" / "mst_inline_example.txt", source="sc.parallelize(...)")
validate_vs_kruskal(res2, edges=inline_edges, name="§2 inline RDD")
pd.DataFrame(res2.mst_edges, columns=["u", "v", "weight"])

✓ §2 inline RDD: Kruskal agrees — weight=9, components=1, |MST|=3


,u,v,weight
0,0,1,4
1,1,2,2
2,2,3,3


## 3. Adjacency list file

`data/sample_adjacency.txt` — same triangle as §1.

In [17]:
adj_path = PROJ_ROOT / "data" / "sample_adjacency.txt"
res_adj = run_boruvka_from_file(sc, str(adj_path), fmt="adjacency_list")
write_mst_file(res_adj, PROJ_ROOT / "output" / "mst_sample_adjacency.txt", source=str(adj_path))
validate_vs_kruskal(
    res_adj, edges_path=adj_path, fmt="adjacency_list", name="§3 sample_adjacency"
)
pd.DataFrame(res_adj.mst_edges, columns=["u", "v", "weight"])

✓ §3 sample_adjacency: Kruskal agrees — weight=3, components=1, |MST|=2


,u,v,weight
0,0,1,1
1,1,2,2


## 4. Disconnected graph (minimum spanning *forest*)

`data/sample_disconnected.txt` — two components, `num_components == 2`.

In [18]:
disc_path = PROJ_ROOT / "data" / "sample_disconnected.txt"
res_disc = run_boruvka_from_file(sc, str(disc_path), fmt="edge_list")
write_mst_file(res_disc, PROJ_ROOT / "output" / "mst_sample_disconnected.txt", source=str(disc_path))
print("edges:", res_disc.mst_edges, "weight:", res_disc.total_weight, "components:", res_disc.num_components)
validate_vs_kruskal(res_disc, edges_path=disc_path, fmt="edge_list", name="§4 disconnected")
pd.DataFrame(res_disc.mst_edges, columns=["u", "v", "weight"])

edges: [(0, 1, 5), (2, 3, 7)] weight: 12 components: 2
✓ §4 disconnected: Kruskal agrees — weight=12, components=2, |MST|=2


,u,v,weight
0,0,1,5
1,2,3,7


## 5. Larger graph (`data/complex_random.txt`)

~120 vertices, ~620 edges. Kruskal validation plus optional cross-check against `data/complex_random.meta`.

In [19]:
complex_path = PROJ_ROOT / "data" / "complex_random.txt"
meta_path = PROJ_ROOT / "data" / "complex_random.meta"

res_big = run_boruvka_from_file(sc, str(complex_path), fmt="edge_list")
write_mst_file(res_big, PROJ_ROOT / "output" / "mst_complex_random.txt", source=str(complex_path))
validate_vs_kruskal(
    res_big, edges_path=complex_path, fmt="edge_list", name="§5 complex_random"
)

if meta_path.exists():
    meta: dict[str, int] = {}
    for line in meta_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split(maxsplit=1)
        if len(parts) != 2:
            continue
        key, val = parts
        if key in (
            "reference_total_weight",
            "reference_components",
            "reference_mst_edges",
        ):
            meta[key] = int(val)
    assert res_big.total_weight == meta["reference_total_weight"]
    print(
        "✓ matches complex_random.meta:",
        meta["reference_total_weight"],
        "weight,",
        meta["reference_components"],
        "components,",
        meta["reference_mst_edges"],
        "MST edges",
    )
else:
    print("(no complex_random.meta — skip file cross-check)")

pd.DataFrame(res_big.mst_edges, columns=["u", "v", "weight"]).head(10)

✓ §5 complex_random: Kruskal agrees — weight=752852, components=1, |MST|=119
✓ matches complex_random.meta: 752852 weight, 1 components, 119 MST edges


,u,v,weight
0,0,42,8574
1,0,44,11889
2,1,14,4932
3,1,92,9448
4,2,42,2059
5,2,101,6099
6,3,18,3029
7,3,82,3987
8,4,37,15171
9,4,58,5772


## 6. Stop Spark

Run this when you are done so JVM processes exit cleanly (especially if you re-run earlier sections often).

In [ ]:
spark.stop()
print("Spark stopped.")